# 05 — Model Training (Phase 3)

Trains and compares Random Forest, XGBoost, LightGBM, and CatBoost —
all as regressors predicting continuous minimum temperature, then
thresholded — per this project's core design decision (regression-then-
threshold beats a direct classifier's recall on rare-event labels; see
`baselines.py` and the Phase 1 results).

**No hyperparameter tuning yet.** These are reasonable default-ish
parameters, not a tuned result — `val.csv` is deliberately unused here
and reserved for a future tuning pass. This notebook establishes whether
tree ensembles beat the Phase 1 baselines at all before spending time
tuning any one of them.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

from models import random_forest_model, xgboost_model, lightgbm_model, catboost_model
from baselines import persistence_predict, log_run
from evaluate import get_continuous_metrics, get_classification_metrics

In [2]:
train = pd.read_csv('../data/processed/train.csv', parse_dates=['DATE'])
val = pd.read_csv('../data/processed/val.csv', parse_dates=['DATE'])
test = pd.read_csv('../data/processed/test.csv', parse_dates=['DATE'])

FROST_LABEL_COL = 'frost_label_broad'
CLASSIFICATION_THRESHOLD = 3.0
DATASET_TAG = '4City_Tier1'  # NOT the same test set as Phase 1's Basel-only baselines

## Feature set for the head-to-head comparison

Restricted to columns that are fully populated across all four cities
(`wind_speed`/`wind_gust`/`cloud_cover`/`sunshine` are missing for at
least one city — see `04_feature_engineering.ipynb`'s missingness check),
so all four models train on an identical, complete feature matrix. Wind's
contribution is investigated separately below, with a model that can
actually tolerate the gaps.

City is one-hot encoded and included — without it, the models would have
to infer which climate they're in purely from weather patterns, which
throws away a signal they should just be given directly.

In [3]:
PRIMARY_FEATURES = [
    'temp_min', 'temp_mean', 'temp_max', 'humidity_pct', 'pressure',
    'global_radiation', 'precipitation', 'dew_point_c', 'dew_point_depression_c',
    'temp_min_lag1', 'temp_min_lag2', 'temp_min_lag3',
    'humidity_pct_lag1', 'humidity_pct_lag2', 'humidity_pct_lag3',
    'temp_min_roll3_min', 'temp_min_roll3_mean', 'temp_min_roll3_max',
    'temp_min_roll7_min', 'temp_min_roll7_mean', 'temp_min_roll7_max',
    'humidity_pct_roll3_min', 'humidity_pct_roll3_mean', 'humidity_pct_roll3_max',
    'humidity_pct_roll7_min', 'humidity_pct_roll7_mean', 'humidity_pct_roll7_max',
    'temp_min_roc1', 'doy_sin', 'doy_cos',
]

train_oh = pd.get_dummies(train, columns=['city'], prefix='city')
test_oh = pd.get_dummies(test, columns=['city'], prefix='city')
CITY_COLS = [c for c in train_oh.columns if c.startswith('city_')]
FEATURES = PRIMARY_FEATURES + CITY_COLS

# The lag/rate-of-change columns have a handful of NaNs at each city's
# start-of-series boundary (see 04's missingness check) -- 12 rows out of
# 10,228, negligible to drop for a clean, identical matrix across models.
train_c = train_oh.dropna(subset=FEATURES + ['target_min_temp'])
test_c = test_oh.dropna(subset=FEATURES + ['target_min_temp'])
print(f"Train: {len(train_c)} rows  Test: {len(test_c)} rows  Features: {len(FEATURES)}")

Train: 10216 rows  Test: 2196 rows  Features: 34


## Fair baseline on the SAME test set

Phase 1's Persistence/Physical/Logistic-regression numbers in
`experiments/runs.csv` were computed on Basel alone -- not directly
comparable to a 4-city result. Re-running Persistence here, on the exact
same `test_c` rows the models below are evaluated on, so the comparison
in `reports/comparison_table.md` is apples-to-apples.

In [4]:
pred_persistence = persistence_predict(test_c['temp_min'])
cont_p = get_continuous_metrics(test_c['target_min_temp'], pred_persistence)
label_p = (pred_persistence <= CLASSIFICATION_THRESHOLD).astype(int)
class_p = get_classification_metrics(test_c[FROST_LABEL_COL], label_p)

print("--- Persistence (4-city test set) ---")
print(f"MAE: {cont_p['mae']:.3f}C  Peirce: {class_p['peirce_skill']:.3f}")

log_run('../experiments/runs.csv', model='Persistence', dataset=DATASET_TAG,
        metrics={**cont_p, **class_p})

--- Persistence (4-city test set) ---
MAE: 2.107C  Peirce: 0.769


## Train and evaluate all four models

In [5]:
MODEL_FACTORIES = {
    'RandomForest': random_forest_model,
    'XGBoost': xgboost_model,
    'LightGBM': lightgbm_model,
    'CatBoost': catboost_model,
}

results = {}
for name, factory in MODEL_FACTORIES.items():
    model = factory().fit(train_c[FEATURES], train_c['target_min_temp'])
    preds = model.predict(test_c[FEATURES])

    cont = get_continuous_metrics(test_c['target_min_temp'], preds)
    pred_label = (preds <= CLASSIFICATION_THRESHOLD).astype(int)
    clazz = get_classification_metrics(test_c[FROST_LABEL_COL], pred_label)

    results[name] = {**cont, **clazz}
    log_run('../experiments/runs.csv', model=name, params=f'{len(FEATURES)} features, default params',
            dataset=DATASET_TAG, metrics=results[name])

pd.DataFrame(results).T.round(3)

,rmse,mae,precision,recall,f1,peirce_skill
RandomForest,2.175,1.715,0.878,0.835,0.856,0.790
XGBoost,2.129,1.681,0.887,0.846,0.866,0.805
LightGBM,2.136,1.680,0.885,0.846,0.865,0.804
CatBoost,2.158,1.713,0.882,0.851,0.866,0.807


All four beat both Persistence and Phase 1's Physical/threshold baseline
on MAE and Peirce skill (see `reports/comparison_table.md` for the full
side-by-side once this is written up there). XGBoost and LightGBM come
out slightly ahead on Peirce skill; differences between the four are
modest at this untuned stage — worth revisiting after a real
hyperparameter search on `val.csv`, not concluding "XGBoost wins"
from one untuned run.

## Feature importance — the deliverable Repo B is waiting on

Uses the FULL feature set (including `wind_speed`, `wind_gust`,
`cloud_cover`, `sunshine` — all with per-city gaps) and a model that
tolerates missing values natively (LightGBM), rather than the
gap-free-only set used for the fair comparison above. This is a
deliberately separate run, not a fifth entry in the model comparison.

**Split-count importance (LightGBM's default) is a known trap here**:
a continuous variable like wind speed gets picked for more splits simply
because it has more distinct values to split on, not necessarily because
it reduces error more. Using gain-based importance instead, which credits
a feature by how much it actually reduced loss when used.

In [6]:
FULL_FEATURES = [
    'temp_min', 'temp_mean', 'temp_max', 'humidity_pct', 'pressure',
    'global_radiation', 'precipitation', 'cloud_cover', 'sunshine',
    'wind_speed', 'wind_gust', 'dew_point_c', 'dew_point_depression_c',
    'temp_min_lag1', 'temp_min_lag2', 'temp_min_lag3',
    'humidity_pct_lag1', 'humidity_pct_lag2', 'humidity_pct_lag3',
    'temp_min_roll3_mean', 'temp_min_roll7_mean',
    'humidity_pct_roll3_mean', 'humidity_pct_roll7_mean',
    'temp_min_roc1', 'doy_sin', 'doy_cos',
] + CITY_COLS

importance_model = lightgbm_model(importance_type='gain').fit(
    train_oh[FULL_FEATURES], train_oh['target_min_temp']
)
importances = importance_model.feature_importances()
importances = importances / importances.sum()  # normalize to proportions of total gain

importances.head(12).round(4)

temp_mean              0.8992
temp_min               0.0185
temp_min_roll7_mean    0.0132
dew_point_c            0.0086
temp_max               0.0056
wind_speed             0.0049
cloud_cover            0.0049
pressure               0.0047
doy_cos                0.0040
temp_min_roc1          0.0040
precipitation          0.0035
doy_sin                0.0035
dtype: float64

In [7]:
wind_rank = importances.rank(ascending=False)[['wind_speed', 'wind_gust']]
print(f"wind_speed: rank {int(wind_rank['wind_speed'])} of {len(importances)}, "
      f"{importances['wind_speed']*100:.2f}% of total gain")
print(f"wind_gust:  rank {int(wind_rank['wind_gust'])} of {len(importances)}, "
      f"{importances['wind_gust']*100:.2f}% of total gain")

wind_speed: rank 6 of 30, 0.49% of total gain
wind_gust:  rank 17 of 30, 0.18% of total gain


**Reading this honestly:**

- `temp_mean` dominates gain-based importance overwhelmingly (~90% in
  this run) — today's mean temperature is extremely predictive of
  tomorrow's minimum on its own, in this climate. This echoes (in a much
  more extreme form) Omazić et al. (2024)'s finding that Tmin alone
  carried about half their model's total importance — Tmin-family
  variables dominating is consistent with the literature, not a red flag.
- `wind_speed` ranks 6th of ~30 features at roughly 0.5% of total gain —
  a real but small contribution. Not nothing, but nowhere near essential.
- **Verdict for Repo B, stated plainly and provisionally:** on this
  Tier-1 European data, a dedicated anemometer looks like a "nice to
  have, not a must-have" — small, measurable signal, dominated by
  temperature history. This should be treated as a first-pass answer,
  not final: Warora's climate and the eventual local threshold (Tier 3)
  may weight wind differently than these mid-latitude European stations
  do, and this result should be re-checked once Tier 3 data exists before
  a final hardware call.

## Next steps

- Hyperparameter tuning on `val.csv` (currently unused) for whichever of
  XGBoost/LightGBM looks most promising after a proper search, not just
  this one untuned run.
- `export_model.py` — serialize the chosen model for Repo B to load.
- Regional calibration on Tier 3 data (`06_regional_calibration.ipynb`)
  before treating any of this — including the wind verdict above — as
  applicable to Warora rather than to mid-latitude Europe.